# 05 — Authenticate, then query

**Access pattern: a credentialed API with a catalogue.** Everything so far was
anonymous. This one is not.

The Copernicus Marine Service is the only source in the workshop that needs an account.
It is also the only one that supplies **currents** — Argo has temperature and salinity
but no velocity, and ERDDAP has surface temperature only — which is why the account is
load-bearing rather than optional.

**If you have not set up an account, this notebook still runs.** It detects that and
teaches the parts that do not need credentials. See section 1.

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

_fetch.OFFLINE and print("offline mode: serving prefetched responses")

offline mode: serving prefetched responses


## 1. Check first, fail gracefully

Account provisioning is slow and somebody always arrives with an unverified address. The
honest design is to detect that and keep teaching, not to raise a `Traceback` at a room
of thirty people.

In [2]:
import os
import shutil
import subprocess
from pathlib import Path

CRED_DIR = Path.home() / ".copernicusmarine"
has_cli = shutil.which("copernicusmarine") is not None
has_creds = CRED_DIR.exists() and any(CRED_DIR.iterdir())

print(f"  copernicusmarine CLI : {'found' if has_cli else 'NOT FOUND'}")
print(f"  credentials in {CRED_DIR} : {'yes' if has_creds else 'no'}")
print()

READY = has_cli and has_creds
if READY:
    print("  -> running the live sections below.")
else:
    print("  -> NOT READY. Sections 3-5 are skipped; 2 and 4 still teach something.")
    print()
    print("  To set this up (takes a few minutes, and an email verification):")
    print("    1. Register: https://data.marine.copernicus.eu/register")
    print("    2. pip install copernicusmarine   (already in this project's deps)")
    print("    3. copernicusmarine login")

  copernicusmarine CLI : found
  credentials in /Users/dev/.copernicusmarine : yes

  -> running the live sections below.


## 2. The catalogue, and a 170 MB mistake

Every Copernicus dataset has a metadata catalogue. Fetch it *filtered*, never bare.

| command | result |
|---|---|
| `describe` with no filter | **170 MB** of JSON, minutes of download |
| `describe --dataset-id <id>` | a few KB |

This is the single most common way this CLI is misused, and it looks like a network
problem rather than a syntax problem.

In [3]:
# Show the flag, not the result -- the bare call is deliberately NOT made here.
cmd = ["copernicusmarine", "describe", "--dataset-id", S.GLORYS_DATASET]
print("  the right way:")
print("   ", " ".join(cmd))
print()
print("  the 170 MB way, which you should never run by accident:")
print("    copernicusmarine describe")
print()
print("  Other verified flags, because the CLI is not internally consistent:")
print("    --end-datetime   NOT --stop-datetime  (the latter is rejected)")
print("    --variable       pass explicitly; the advertised variable count is 0")

  the right way:
    copernicusmarine describe --dataset-id GLOBAL_MULTIYEAR_PHY_001_030

  the 170 MB way, which you should never run by accident:
    copernicusmarine describe

  Other verified flags, because the CLI is not internally consistent:
    --end-datetime   NOT --stop-datetime  (the latter is rejected)
    --variable       pass explicitly; the advertised variable count is 0


### ⚠️ Trap — the advertised variable count is `0`

`copernicusmarine describe` reports the number of variables in a dataset as **zero**,
for datasets that plainly have variables. A correct dataset looks empty.

The consequence is specific: a script that discovers variables by asking the catalogue
gets nothing, and quietly requests no data. Do not use the summary to build a request —
name the variables you want.

In [4]:
VARIABLES = ["thetao", "so", "uo", "vo"]   # temperature, salinity, u, v
print("  the request is built from variables WE name, not from what the catalogue says:")
print("   ", ", ".join(VARIABLES))
print()
print("  GLORYS short names, and what they mean:")
for v, meaning in zip(VARIABLES, ["potential temperature", "salinity",
                                  "eastward current", "northward current"]):
    print(f"    {v:8} {meaning}")
print()
print("  Our own project renames these to temperature/salinity/u_eastward/v_northward,")
print("  so the rest of the code never has to know the vendor's short names.")

  the request is built from variables WE name, not from what the catalogue says:
    thetao, so, uo, vo

  GLORYS short names, and what they mean:
    thetao   potential temperature
    so       salinity
    uo       eastward current
    vo       northward current

  Our own project renames these to temperature/salinity/u_eastward/v_northward,
  so the rest of the code never has to know the vendor's short names.


## 3. A subset request

Bounding the request in **all** six dimensions is what makes it fast. The dataset is
global at 1/12° with 50 levels and daily resolution from 1993; the box is 0.22° square
and 60 m deep, for a two-year window.

In [5]:
from ocean_sim.config import OCEAN_BOX

box = OCEAN_BOX
subset = [
    "copernicusmarine", "subset",
    "--dataset-id", S.GLORYS_DATASET,
    "--variable", "thetao",
    "--minimum-longitude", str(box["min_longitude"]),
    "--maximum-longitude", str(box["max_longitude"]),
    "--minimum-latitude",  str(box["min_latitude"]),
    "--maximum-latitude",  str(box["max_latitude"]),
    "--minimum-depth",     str(box["min_depth"]),
    "--maximum-depth",     str(box["max_depth"]),
    "--start-datetime", "2019-01-01T00:00:00",
    "--end-datetime",   "2021-05-01T00:00:00",
    "--output-directory", "data/glorys",
    "--file-format", "netcdf",
]
for part in subset:
    print("   ", part)
print()
print("  Six bounds plus a time window. Omit any one and the request is global:")
print("  that is the difference between a 30-second call and a stalled workshop.")

    copernicusmarine
    subset
    --dataset-id
    GLOBAL_MULTIYEAR_PHY_001_030
    --variable
    thetao
    --minimum-longitude
    -122.2
    --maximum-longitude
    -121.98
    --minimum-latitude
    36.6
    --maximum-latitude
    36.82
    --minimum-depth
    0.0
    --maximum-depth
    60.0
    --start-datetime
    2019-01-01T00:00:00
    --end-datetime
    2021-05-01T00:00:00
    --output-directory
    data/glorys
    --file-format
    netcdf

  Six bounds plus a time window. Omit any one and the request is global:
  that is the difference between a 30-second call and a stalled workshop.


In [6]:
if READY:
    print("  running the live subset -- this is the slow cell in the notebook...")
    r = subprocess.run(subset + ["--overwrite"], capture_output=True, text=True, timeout=1800)
    print("  exit:", r.returncode)
    print((r.stdout or r.stderr)[-600:])
else:
    print("  SKIPPED -- no credentials on this machine.")
    print()
    print("  What you would have got, from the project's own loaded copy:")
    print("    30 days x 19 levels over 0.5-55.8 m, surface 15.62 C")
    print("    2019-2021: 16,188 rows in ocean_profile_daily")

  running the live subset -- this is the slow cell in the notebook...


  exit: 1
ERROR - 2026-09-29T03:51:17Z - Dataset not found: GLOBAL_MULTIYEAR_PHY_001_030 Please check that the dataset exists and the input datasetID is correct.



## 4. Why a fallback ladder is not indecision

GLORYS is the only source for currents, and it is the only one that can fail for a
reason no amount of retrying fixes: **you do not have an account.**

That is a different failure from a timeout, and it should be planned for rather than
hoped against. This project therefore verified ten sources and built a ladder:

| tier | sources | if GLORYS is unavailable |
|---|---|---|
| 1 | GLORYS, Argo, ERDDAP SST, NDBC | — |
| 2 | WOA23 climatology, GEBCO bathymetry, HYCOM | temperature, salinity, currents from other providers |
| 3 | OBIS, Orcasound, ShipsEar, Watkins | biological and acoustic context |

A workshop that depends on one credentialed source is a workshop with a single point of
failure. This is the same reason the notebooks degrade to cache rather than raising: the
design assumption throughout is that *something* will be unavailable.

## 5. Handling credentials

Two rules, both learned the hard way:

1. **Never commit them.** This project keeps credentials in `~/.copernicusmarine/` and a
   gitignored `.env`, and no credential has ever been committed.
2. **Never share a laptop that has them.** A cached Copernicus login is a persistent
   credential on someone else's machine. This is the reason the workshop's prefetch
   deliberately does **not** cache anything from this notebook — every other source is
   anonymous, and caching those is harmless.

In [7]:
print("  readiness:", "LIVE" if READY else "SKIPPED (no credentials)")
print()
print("  For reference, the dataset this notebook targets:")
print("   ", S.COPERNICUS_HOME)
print()
print("  Product page, dataset id, and the catalogue of every variable are all public")
print("  and need no account -- it is only the data download that is credentialed.")

  readiness: LIVE

  For reference, the dataset this notebook targets:
    https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/description

  Product page, dataset id, and the catalogue of every variable are all public
  and need no account -- it is only the data download that is credentialed.
